# GP04 — Explore GeoPackage

## Στόχος, είσοδος και παραδοτέο

Θα ανοίξεις το GeoPackage `gp04/serres_assets.gpkg`, θα ανακαλύψεις τα layers του, θα ελέγξεις χωριστά τα `parks` και `water_points` και θα εξαγάγεις ένα ελεγμένο αποτέλεσμα. Στο τέλος παρέδωσε τα βασικά ευρήματα για rows, IDs, schema, CRS, geometry και read-back του export.

Χρειάζεσαι τις βασικές έννοιες DataFrame, συνάρτησης και ελέγχου με `assert` από το GP00–GP03. Το GeoPackage είναι container με ονομασμένα layers· κάθε layer μπορεί να έχει διαφορετικό schema ή geometry type, γι’ αυτό δεν αρκεί ένας συνολικός έλεγχος.

### Προετοιμασία δεδομένων — **Εκτέλεσε πρώτα — μη μεταβάλεις**

Το επόμενο κελί προετοιμάζει και ελέγχει το dataset `gp04/serres_assets.gpkg`. Το μήνυμα `[OK] Using validated dataset root:` σημαίνει ότι βρέθηκε ήδη διαθέσιμος και βασικά επικυρωμένος φάκελος δεδομένων. Αν η προετοιμασία αποτύχει με `Could not prepare dataset`, σταμάτησε και έλεγξε την πρόσβαση/λήψη δεδομένων· δεν είναι αποτυχία της δικής σου ανάλυσης.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp04'
DATA_URLS = {'serres_assets.gpkg': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp04/serres_assets.gpkg'}
SCHEMA = {'serres_assets.gpkg': {'kind': 'gpkg', 'layers': ['parks', 'water_points']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP04 — Explore GeoPackage

Ανοίγω → Εξερευνώ → Κατανοώ → Φιλτράρω → Επαληθεύω → Εξάγω

Το GeoPackage είναι container, όχι απαραίτητα ένα layer. Ανακάλυψε πρώτα τα layer names και έπειτα έλεγξε κάθε layer χωριστά: schema, attributes, CRS, geometry, bounds και map.


In [ ]:
import geopandas as gpd
from pathlib import Path
gpkg_path = DATA_ROOT / "gp04" / "serres_assets.gpkg"
assert gpkg_path.exists()


## 1. Layers

Πρώτα ανακαλύπτουμε τι περιέχει το GeoPackage. Το `gpd.list_layers()` διαβάζει τον κατάλογο των layers χωρίς να φορτώσει ακόμη όλα τα features. Αν δεν υποστηρίζεται στο περιβάλλον σου, κατέγραψε το και ακολούθησε μόνο τεκμηριωμένο fallback.

**Checkpoint:** Ποια είναι τα πραγματικά layer names; Γιατί είναι χρήσιμο να τα γνωρίζεις πριν καλέσεις `read_file()`;

In [ ]:
# TODO: χρησιμοποίησε gpd.list_layers(gpkg_path) και επιβεβαίωσε τα πραγματικά layer names.
# TODO: αν το περιβάλλον δεν υποστηρίζει list_layers(), κατέγραψε το και χρησιμοποίησε τεκμηριωμένο fallback.


## 2. Ανάγνωση layer και schema

Διάβασε χωριστά τα `parks` και `water_points` ως `parks` και `water`. Αναμένουμε 3 Polygon features στο `parks` και 3 Point features στο `water_points`. Έλεγξε rows, columns, dtypes και missing values ανά layer: το ίδιο GeoPackage δεν σημαίνει ίδιο schema.

**Checkpoint:** Ποια πεδία είναι κοινά και ποια layer-specific; Ποιον έλεγχο θα επαναλάμβανες αν ένα layer είχε απρόσμενο τύπο γεωμετρίας;

In [ ]:
# TODO: διάβασε χωριστά τα parks και water_points ως parks και water.
# TODO: εμφάνισε rows, columns, dtypes και missing values για κάθε layer.


## 3. CRS, geometry types και bounds

Τώρα ελέγχεις τη χωρική ερμηνεία κάθε layer. Τα `geom_type`, `total_bounds` και `is_valid` δείχνουν τύπο γεωμετρίας, χωρική έκταση και βασική εγκυρότητα· κανένας από αυτούς τους ελέγχους μόνος του δεν αποδεικνύει ότι τα δεδομένα είναι σημασιολογικά σωστά.

Στο plot παρατήρησε αν τα σημεία και τα πάρκα βρίσκονται στην ίδια χωρική περιοχή.

In [ ]:
# TODO: έλεγξε CRS, geom_type.value_counts(), total_bounds και is_valid για κάθε layer.
# TODO: κάνε plot τα δύο layers στον ίδιο άξονα.


## 4. Filtering και QA

Το guided filtering εφαρμόζει το `status == 'active'` στα water points και συνδυάζει το φίλτρο με ελέγχους layer-specific. Κατέγραψε rows, IDs, geometry type και CRS πριν από το export. Ένα επιτυχημένο `to_file()` αποδεικνύει μόνο ότι γράφτηκε αρχείο· δεν αποδεικνύει ότι γράφτηκε το σωστό αποτέλεσμα, γι’ αυτό χρειάζεται read-back.

In [ ]:
# TODO: δημιούργησε μόνος/μόνη σου τη μεταβλητή active_water, φιλτράροντας status == 'active'.
# TODO: πρόσθεσε layer-specific checks και κάνε export σε GeoJSON.


## Mini Challenge

Επανάλαβε ανεξάρτητα τη σχετική ροή για τα active water points, χωρίς να αντιγράψεις την προηγούμενη απάντηση. Τεκμηρίωσε rows, IDs, geometry και CRS, εξήγαγε το αποτέλεσμα και διάβασέ το ξανά για να επαληθεύσεις ότι το αρχείο περιέχει ακριβώς το ζητούμενο αποτέλεσμα.

In [ ]:
# Γράψε εδώ τη λύση σου
